# 06 — the EXPLAIN comparison workbook

Builds `reports/api-explain-<case>-<stamp>.xlsx`, **one file per case**, and
copies each to `api-explain-<case>-latest.xlsx`.

**Open the `-latest` one.** Stamped names accumulate and most file pickers
sort them by stamp rather than by recency, so the oldest generation sits at
the top of the list — and a stale workbook is internally consistent and says
nothing about being stale. `Provenance` row 1 names the build it came from if
you need to check.

The markdown reports are the narrative; this is the instrument. 1,414 statements
across three identities cannot be compared by scrolling — a plan rendered beside
its statement is right for reading one API and useless for asking which
statements differ.

Design and the reasoning behind each sheet: `PLAN-explain-workbook.md`.

**Touches no database and no cluster.** Two files in, one workbook out.

## Sheets

| sheet | one row per | answers |
|---|---|---|
| `Summary` | API | which operations seq-scan, and on what |
| `Statements` | statement occurrence | what one API issues, in order |
| `Distinct` | (SQL, params) pair | what the sweep planned, with weight |
| `Cost` | (SQL, params) pair | **which statement is expensive, and next to what** |
| `Shapes` | logical statement | near-duplicates collapsed by predicate SET |
| `Unplanned` | refusal | redacted (permanent) vs refused (recoverable) |
| `Schema` | live index | what exists, and what upstream defines |
| `Provenance` | — | sources, volumes, and what NOT to do with the numbers |

## The `Cost` sheet — rows and cost, side by side

`rows_scanned` is what the plan **reads** from its table: the whole table for a
Seq Scan, the `Index Cond` match for an index node. It is **not** `plan_rows`,
which on a Seq Scan is what *survives* the `Filter` and reads `1` on a
60,815-row table.

**No ratio, on purpose.** Rows and cost sit next to each other and you draw the
line. They are not proportional — 60,815 rows read sequentially costs about
194× a single-row index seek, not 60,815×, because a sequential read is far
cheaper *per row* than the same number of random index descents.

`cost_units` is `total_cost × occurrences`, `cost_share` its fraction of the
case. Planner units, not milliseconds, within-case only.

## Before the 10,000-principal reseed: sort on `scales_with_table`

`TRUE` means the plan reads the **whole relation** — both its rows and its cost
move with the row count. Seeding multiplies those; the `FALSE` rows stay where
they are. That split is what forecasts, because plan *shape* is the part plain
`EXPLAIN` measures reliably.

Today, at 60,815 `grant_records`:

| case | TRUE statements | occurrences | share of case cost |
|---|---:|---:|---:|
| unauthorized | 2 of 33 | 52 | 95.5% |
| authorized | 6 of 91 | 54 | 94.1% |
| admin | 11 of 128 | 67 | 93.8% |

Every one of them is on `grant_records`.

Two cautions before extrapolating. A shape can **flip** at volume — an
unindexed scan may escalate to a *parallel* Seq Scan past roughly 160k rows,
which is a different regime rather than more of this one. And growth is not
uniform: `grant_records` grows with principals × grants, `entities` with
catalogs and tables. **Re-run the sweep after the reseed** and compare the two
`Cost` sheets; do not multiply these numbers.

## Read `Provenance` before doing arithmetic

`total_cost` and `plan_rows` are planner **estimates**. Two sweeps 25 minutes
apart differed in 11 of 264 `plan_rows` while **0 of 264** differed in plan
*shape*. Sort and compare shape; do not average, sum or trend the estimates.
`duration_ms` is from the capture, not the EXPLAIN — plain `EXPLAIN` does not
execute.


In [11]:
# --- cell 1: bootstrap ------------------------------------------------------
import gc
import pathlib
import subprocess
import sys

HERE = pathlib.Path.cwd()
REPORTS = HERE / "reports"
RUNS = HERE / "runs"
print(f"reports: {REPORTS}")


reports: /Users/kade/hynix/polaris-practice/polaris-learning/diagnostics/api-sql-profile/reports


## Inputs — set these by hand

One entry per case. Pair them deliberately: a report and a run from different
sweeps still join on most statements and are wrong about all of them. The
generator refuses a run whose `case` or `matrix` disagrees, but it cannot catch
two files that are merely from different *drives* of the same case.


In [12]:
# --- cell 2: INPUTS ---------------------------------------------------------
BUILD = {
    "unauthorized": {
        "matrix": "doc-api-sql-matrix-unauthorized-explained-20260903-141115.md",
        "explain": "apiexplain-unauthorized-20260903-142053.json",
    },
    "authorized": {
        "matrix": "doc-api-sql-matrix-authorized-explained-20260903-141115.md",
        "explain": "apiexplain-authorized-20260903-142055.json",
    },
    "admin": {
        "matrix": "doc-api-sql-matrix-admin-explained-20260903-141115.md",
        "explain": "apiexplain-admin-20260903-142056.json",
    },
}
for case, io in BUILD.items():
    assert (REPORTS / io["matrix"]).exists(), f"{case}: no {io['matrix']}"
    assert (RUNS / io["explain"]).exists(), f"{case}: no {io['explain']}"
    print(f"{case:<13} {io['matrix']}\n{'':<13} + {io['explain']}")


unauthorized  doc-api-sql-matrix-unauthorized-explained-20260903-141115.md
              + apiexplain-unauthorized-20260903-142053.json
authorized    doc-api-sql-matrix-authorized-explained-20260903-141115.md
              + apiexplain-authorized-20260903-142055.json
admin         doc-api-sql-matrix-admin-explained-20260903-141115.md
              + apiexplain-admin-20260903-142056.json


In [13]:
# --- cell 3: build ----------------------------------------------------------
BUILT = {}
for case, io in BUILD.items():
    print(f"\n=== {case} ===")
    p = subprocess.run(
        [sys.executable, "render_explain_workbook.py", "--case", case,
         "--matrix", f"reports/{io['matrix']}",
         "--explain", f"runs/{io['explain']}"],
        cwd=str(HERE), capture_output=True, text=True,
    )
    print(p.stdout.rstrip() or p.stderr.rstrip())
    assert p.returncode == 0, f"{case}: generator exited {p.returncode}"
    BUILT[case] = p.stdout.splitlines()[0].split("-> ")[-1].strip()



=== unauthorized ===
-> api-explain-unauthorized-20260903-142716.xlsx
   summary: 43
   statements: 390
   distinct: 33
   shapes: 6

=== authorized ===
-> api-explain-authorized-20260903-142716.xlsx
   summary: 43
   statements: 460
   distinct: 93
   shapes: 18

=== admin ===
-> api-explain-admin-20260903-142717.xlsx
   summary: 43
   statements: 564
   distinct: 138
   shapes: 21


## Check the counts against the reports

The generator reads the same files the reports were rendered from, so these must
reconcile. A silent mismatch means the wrong pair was named above.


In [14]:
# --- cell 4: reconcile ------------------------------------------------------
import json
import re

for case, io in BUILD.items():
    text = (REPORTS / io["matrix"]).read_text(encoding="utf-8")
    run = json.loads((RUNS / io["explain"]).read_text())
    print(f"{case:<13} report statements={len(re.findall(r'```sql', text)):<5} "
          f"run pairs={run['pairs']:<5} index={run['index_state']:<8} "
          f"grant_records={(run.get('volume') or {}).get('grant_records'):,}")
    #: 43 in every case, in every identity -- the surface is fixed and only the
    #: outcomes change. A different number here means a partial drive.
    assert len(run["per_api"]) == 43, f"{case}: {len(run['per_api'])} APIs, expected 43"


unauthorized  report statements=390   run pairs=33    index=absent   grant_records=60,815
authorized    report statements=460   run pairs=93    index=absent   grant_records=60,815
admin         report statements=564   run pairs=138   index=absent   grant_records=60,815


In [ ]:
# --- cell 5: what was written -----------------------------------------------
for case, name in BUILT.items():
    f = REPORTS / name
    latest = REPORTS / f"api-explain-{case}-latest.xlsx"
    print(f"{case:<13} {name}  ({f.stat().st_size / 1024:.0f} KB)")
    print(f"{'':<13} -> {latest.name}  {'ok' if latest.exists() else 'MISSING'}")
print("\nOpen the -latest copy. Provenance first — it names the sources and "
      "says what the estimates may not be used for.")


In [16]:
# --- cell 5b: rows and cost, without opening Excel --------------------------
#: The Cost sheet's top rows plus the reseed split. Printed here because the
#: finding is a handful of numbers per case and asking someone to open three
#: workbooks to see them loses it.
from openpyxl import load_workbook

for case, name in BUILT.items():
    wb = load_workbook(REPORTS / name, read_only=True)
    it = wb["Cost"].iter_rows(values_only=True)
    h = {k: i for i, k in enumerate(next(it))}
    rows = list(it)
    print(f"\n=== {case} ===")
    print(f"{'table':<15}{'verb':<8}{'occ':>4}{'rows_scanned':>14}  "
          f"{'rows_basis':<26}{'scales':>7}{'cost':>10}{'units':>11}{'share':>8}")
    for r in rows[:6]:
        print(f"{r[h['table']]:<15}{r[h['verb']]:<8}{r[h['occurrences']]:>4}"
              f"{r[h['rows_scanned']]:>14,}  {str(r[h['rows_basis']]):<26}"
              f"{str(r[h['scales_with_table']]):>7}{r[h['total_cost']]:>10}"
              f"{r[h['cost_units']]:>11,.0f}{r[h['cost_share']]:>7.1%}")
    #: The reseed line. These are the statements whose numbers move when
    #: grant_records grows; everything else stays where it is.
    sc = [r for r in rows if r[h["scales_with_table"]]]
    print(f"  reads the whole table: {len(sc)} of {len(rows)} statements, "
          f"{sum(r[h['occurrences']] for r in sc)} occurrences, "
          f"{sum(r[h['cost_share']] for r in sc):.1%} of case cost"
          f"  -> {', '.join(sorted({r[h['table']] for r in sc})) or 'none'}")
    wb.close()



=== unauthorized ===
table          verb     occ  rows_scanned  rows_basis                 scales      cost      units   share
grant_records  SELECT    51        60,815  whole table                  True   1637.26     83,500  93.7%
grant_records  SELECT     1        60,815  whole table                  True   1637.26      1,637   1.8%
entities       SELECT    15             2  index match, lower bound    False     40.02        600   0.7%
entities       SELECT    17             2  index match, lower bound    False      32.1        546   0.6%
entities       SELECT    50             1  index match                 False      8.31        416   0.5%
entities       SELECT    50             1  index match, lower bound    False      8.31        416   0.5%
  reads the whole table: 2 of 33 statements, 52 occurrences, 95.5% of case cost  -> grant_records

=== authorized ===
table          verb     occ  rows_scanned  rows_basis                 scales      cost      units   share
grant_records  SEL

In [17]:
# --- cell 6: teardown -------------------------------------------------------
for _n in ("BUILT",):
    if _n in dir():
        del globals()[_n]
gc.collect()
print("done")


done
